In [ ]:
import json
import re
import os
from pathlib import Path
import pandas as pd

# RESULT_FOLDER_PATH = "/Users/lockewang/FIG/WebDomainRandomizer/baseline_results_new/baseline_results"
RESULT_FOLDER_PATH = "/Users/lockewang/FIG/WebDomainRandomizer/baseline_results_full_new/baseline_results"

pd.set_option('display.max_colwidth', None)

# Loading predictions

In [ ]:
# load predictions from jsonl files for each model, reasoning_type, query_type combo into one dataframe

# Find all .jsonl files in the result folder
jsonl_files = sorted(list(Path(RESULT_FOLDER_PATH).glob("*.jsonl")))
print(f"Found {len(jsonl_files)} .jsonl files")

# Load all jsonl files into a list of dataframes
dataframes = {}
for jsonl_file in jsonl_files:
    print(f"Loading {jsonl_file.name}")
    df = pd.read_json(jsonl_file, lines=True)
    dataframes[jsonl_file.name] = df


In [ ]:
gta1_dfs = []
for filename, df in dataframes.items():
    if filename.startswith("predictions_gta1"):
        gta1_dfs.append(df)

# Concatenate all GTA1 DataFrames
if gta1_dfs:
    df_all = pd.concat(gta1_dfs, ignore_index=True)
else:
    raise ValueError("No GTA1 DataFrames found")


# Stitch together re-runs due to the `nan` relational query error at row 1625.
configs = [
    ("predictions_qwen25vl_no_reasoning_direct_query_20251214_022358", "predictions_qwen25vl_no_reasoning_direct_query_20251215_021838", "replace"),
    ("predictions_qwen25vl_no_reasoning_relational_query_20251214_022351", "predictions_qwen25vl_no_reasoning_relational_query_20251215_021833", "concat"),
    ("predictions_qwen25vl_reasoning_direct_query_20251214_022355", "predictions_qwen25vl_reasoning_direct_query_20251215_021836", "replace"),
    ("predictions_qwen25vl_reasoning_relational_query_20251214_022344", "predictions_qwen25vl_reasoning_relational_query_20251215_021829", "concat"),
    ("predictions_uitars15_no_reasoning_direct_query_20251213_223027", "predictions_uitars15_no_reasoning_direct_query_20251215_024416", "replace"),
    ("predictions_uitars15_no_reasoning_relational_query_20251213_222956", "predictions_uitars15_no_reasoning_relational_query_20251215_024414", "replace"),
    ("predictions_uitars15_reasoning_direct_query_20251213_223013", "predictions_uitars15_reasoning_direct_query_20251215_024415", "replace"),
    ("predictions_uitars15_reasoning_relational_query_20251213_222938", "predictions_uitars15_reasoning_relational_query_20251215_024412", "replace")
]

for config in configs:
    big_df = dataframes[config[0]+".jsonl"].copy()
    small_df = dataframes[config[1]+".jsonl"].copy()
    if config[2] == "replace":
        big_df.iloc[-92:] = small_df.values
    elif config[2] == "concat":
        big_df = pd.concat([big_df, small_df], ignore_index=True)

    print(f"config: {config}")
    print(f"len(big_df): {len(big_df)}")
    df_all = pd.concat([df_all, big_df], ignore_index=True)

"""
Processing predictions_gta1_no_reasoning_direct_query_20251214_220352.jsonl (1/20)
1716
Processing predictions_gta1_no_reasoning_relational_query_20251214_220408.jsonl (2/20)
1716
Processing predictions_gta1_reasoning_direct_query_20251214_220400.jsonl (3/20)
1716
Processing predictions_gta1_reasoning_relational_query_20251214_220417.jsonl (4/20)
1716

Processing predictions_qwen25vl_no_reasoning_direct_query_20251214_022358.jsonl (5/20)
1716
Processing predictions_qwen25vl_no_reasoning_direct_query_20251215_021838.jsonl (6/20)
92

Processing predictions_qwen25vl_no_reasoning_relational_query_20251214_022351.jsonl (7/20)
1624
Processing predictions_qwen25vl_no_reasoning_relational_query_20251215_021833.jsonl (8/20)
92

Processing predictions_qwen25vl_reasoning_direct_query_20251214_022355.jsonl (9/20)
1716
Processing predictions_qwen25vl_reasoning_direct_query_20251215_021836.jsonl (10/20)
92

Processing predictions_qwen25vl_reasoning_relational_query_20251214_022344.jsonl (11/20)
1624
Processing predictions_qwen25vl_reasoning_relational_query_20251215_021829.jsonl (12/20)
92

Processing predictions_uitars15_no_reasoning_direct_query_20251213_223027.jsonl (13/20)
1716
Processing predictions_uitars15_no_reasoning_direct_query_20251215_024416.jsonl (14/20)
92

Processing predictions_uitars15_no_reasoning_relational_query_20251213_222956.jsonl (15/20)
1716
Processing predictions_uitars15_no_reasoning_relational_query_20251215_024414.jsonl (16/20)
92

Processing predictions_uitars15_reasoning_direct_query_20251213_223013.jsonl (17/20)
1716
Processing predictions_uitars15_reasoning_direct_query_20251215_024415.jsonl (18/20)
92

Processing predictions_uitars15_reasoning_relational_query_20251213_222938.jsonl (19/20)
1716
Processing predictions_uitars15_reasoning_relational_query_20251215_024412.jsonl (20/20)
92

"""

In [ ]:
# # load predictions from jsonl files for each model, reasoning_type, query_type combo into one dataframe

# # Find all .jsonl files in the result folder
# jsonl_files = list(Path(RESULT_FOLDER_PATH).glob("*.jsonl"))
# print(f"Found {len(jsonl_files)} .jsonl files")

# # Load all jsonl files into a list of dataframes
# dataframes = []
# for jsonl_file in jsonl_files:
#     print(f"Loading {jsonl_file.name}")
#     df = pd.read_json(jsonl_file, lines=True)
#     dataframes.append(df)

# # Concatenate all dataframes into one
# df_all = pd.concat(dataframes, ignore_index=True)

print(f"\nTotal rows loaded: {len(df_all)}")
print(f"\nColumns: {df_all.columns.tolist()}")
print(f"\nFirst few rows:")
df_all.head()
df_all.info()
df_all.isna().sum()
pd.set_option('display.max_rows', None)
# df_all.groupby(["model", "use_reasoning", "query_type"]).value_counts()


In [ ]:
# pd.concat([gta1_df, qwen_df, uitars15_df]).to_csv('baseline_results_combined.csv', index=False)
df_all.to_csv('baseline_results_full_new.csv', index=False)


# Parse raw predictions into actions for each model

## Common helper code

In [ ]:
import math

IMAGE_FACTOR = 28
MIN_PIXELS = 100 * 28 * 28
MAX_PIXELS = 16384 * 28 * 28
MAX_RATIO = 200
IMAGE_HEIGHT = 1080
IMAGE_WIDTH = 1920

def escape_single_quotes(text):
    # 匹配未转义的单引号（不匹配 \\'）
    pattern = r"(?<!\\)'"
    return re.sub(pattern, r"\\'", text)

def round_by_factor(number: int, factor: int) -> int:
    """Returns the closest integer to 'number' that is divisible by 'factor'."""
    return round(number / factor) * factor


def ceil_by_factor(number: int, factor: int) -> int:
    """Returns the smallest integer greater than or equal to 'number' that is divisible by 'factor'."""
    return math.ceil(number / factor) * factor


def floor_by_factor(number: int, factor: int) -> int:
    """Returns the largest integer less than or equal to 'number' that is divisible by 'factor'."""
    return math.floor(number / factor) * factor    


def smart_resize(
    height: int, width: int, factor: int = IMAGE_FACTOR, min_pixels: int = MIN_PIXELS, max_pixels: int = MAX_PIXELS
) -> tuple[int, int]:
    """
    Rescales the image so that the following conditions are met:

    1. Both dimensions (height and width) are divisible by 'factor'.

    2. The total number of pixels is within the range ['min_pixels', 'max_pixels'].

    3. The aspect ratio of the image is maintained as closely as possible.
    """
    if max(height, width) / min(height, width) > MAX_RATIO:
        raise ValueError(
            f"absolute aspect ratio must be smaller than {MAX_RATIO}, got {max(height, width) / min(height, width)}"
        )
    h_bar = max(factor, round_by_factor(height, factor))
    w_bar = max(factor, round_by_factor(width, factor))
    if h_bar * w_bar > max_pixels:
        beta = math.sqrt((height * width) / max_pixels)
        h_bar = floor_by_factor(height / beta, factor)
        w_bar = floor_by_factor(width / beta, factor)
    elif h_bar * w_bar < min_pixels:
        beta = math.sqrt(min_pixels / (height * width))
        h_bar = ceil_by_factor(height * beta, factor)
        w_bar = ceil_by_factor(width * beta, factor)
    return h_bar, w_bar


## Qwen2.5VL

Need to parse for number of <tool_call> </tool_tool>, each is an action. And then parse for arguments in
```
<tool_call>
{"name": "computer_use", "arguments": {"action": "left_click", "coordinate": [966, 546]}}
</tool_call>
```
----
```
Thought: I need to locate the textbox labeled 'To Los Angeles (LAX)' and replace 'Los Angeles (LAX)' with 'SAN FRANCISCO'. The textbox is clearly visible and identifiable.
<tool_call>
{"name": "computer_use", "arguments": {"action": "left_click", "coordinate": [704, 468]}}
</tool_call>
<tool_call>
{"name": "computer_use", "arguments": {"action": "type", "text": "SAN FRANCISCO"}}
</tool_call>

 addCriterion
```

- for rows with multiple actions, parse the coordinates from the first action with coordinates as the coordinates
- for rows with multiple actions, parse the first non-click action as the action type. for example, click, type, finished(), should have type as its action type for the row. Another example, call_user() should have 'call_user' as the action type
- browser_select_option get converted to 'select' action




In [ ]:

def parse_qwen_action(response):
    # Calculate the real image size sent into the model
    resized_height, resized_width = smart_resize(
        IMAGE_HEIGHT,
        IMAGE_WIDTH,
        factor=IMAGE_FACTOR,
        min_pixels=MIN_PIXELS,
        max_pixels=MAX_PIXELS,
    )
    
    result_dict = {
        "result": "positive",
        "format": "x1y1x2y2",
        "raw_response": response,
        "bbox": None,
        "point": None,
        "coordinates": None,
        "action_type": None
    }

    # Check if response already has tool_call tags
    if '<tool_call>' not in response:
        # Fallback to old logic: add guide_text
        guide_text = "<tool_call>\n{\"name\": \"computer_use\", \"arguments\": {\"action\": \"left_click\", \"coordinate\": ["
        response = guide_text + response
        cut_index = response.rfind('}')
        if cut_index != -1:
            response = response[:cut_index + 1]
    
    # Find all <tool_call>...</tool_call> blocks
    tool_call_pattern = r'<tool_call>\s*(.*?)\s*</tool_call>'
    tool_calls = re.findall(tool_call_pattern, response, re.DOTALL)
    
    if not tool_calls:
        return result_dict
    
    # Parse all actions
    actions = []
    # Valid click actions from the prompt enum
    click_actions = {'left_click', 'right_click', 'middle_click', 'double_click', 'left_click_drag'}
    # All valid actions from the prompt enum
    valid_actions = {'key', 'type', 'mouse_move', 'left_click', 'left_click_drag', 
                     'right_click', 'middle_click', 'double_click', 'scroll', 'wait', 'terminate'}
    
    for tool_call_str in tool_calls:
        try:
            action = json.loads(tool_call_str.strip())
            if 'arguments' in action:
                actions.append(action)
        except (json.JSONDecodeError, ValueError, TypeError):
            continue
    
    if not actions:
        return result_dict
    
    # Extract coordinates from the first action that has coordinates
    # Note: coordinate is required for mouse_move and left_click_drag, but may also appear with left_click
    for action in actions:
        if 'arguments' in action and isinstance(action['arguments'], dict):
            if 'coordinate' in action['arguments']:
                coordinates = action['arguments']['coordinate']
                if isinstance(coordinates, list) and len(coordinates) >= 2:
                    if len(coordinates) == 2:
                        point_x, point_y = float(coordinates[0]), float(coordinates[1])
                    elif len(coordinates) == 4:
                        x1, y1, x2, y2 = float(coordinates[0]), float(coordinates[1]), float(coordinates[2]), float(coordinates[3])
                        point_x = (x1 + x2) / 2
                        point_y = (y1 + y2) / 2
                    else:
                        continue
                    # Normalize and denormalize predicted coordinates into image dimensions
                    result_dict["point"] = [point_x / resized_width * IMAGE_WIDTH, point_y / resized_height * IMAGE_HEIGHT]
                    result_dict["coordinates"] = [point_x / resized_width * IMAGE_WIDTH, point_y / resized_height * IMAGE_HEIGHT]
                    break
    
    # Extract action type: first non-click action, or first action if all are clicks
    for action in actions:
        if 'arguments' in action and isinstance(action['arguments'], dict):
            if 'action' in action['arguments']:
                action_name = action['arguments']['action']
                # Validate it's a valid action
                if action_name in valid_actions:
                    # If it's not a click action, use it
                    if action_name not in click_actions:
                        result_dict["action_type"] = action_name
                        return result_dict
    
    # If all actions were clicks (or no valid action found), use the first action's type
    if actions:
        first_action = actions[0]
        if 'arguments' in first_action and isinstance(first_action['arguments'], dict):
            if 'action' in first_action['arguments']:
                action_name = first_action['arguments']['action']
                if action_name in valid_actions:
                    result_dict["action_type"] = action_name
    
    return result_dict


In [ ]:
import re
import json
from typing import Tuple, Optional, List

def parse_qwen_prediction(raw_prediction: str, debug: bool = True) -> Tuple[Optional[str], Optional[List[int]]]:
    """
    Parse action type and coordinates from Qwen2.5VL raw predictions.
    
    Args:
        raw_prediction: Raw text prediction from Qwen2.5VL model
        debug: If True, print debug information
        
    Returns:
        Tuple of (action_type, coordinates) where:
        - action_type: str or None (e.g., 'type', 'call_user', 'left_click', etc.)
        - coordinates: [x, y] list or None
        
    Special handling:
    - For multiple actions: extracts coordinates from first action with coordinates
    - For multiple actions: returns first non-click action type (e.g., 'type', 'call_user')
    - If only click actions exist, returns the first click action type
    - Returns (None, None) for predictions without valid tool calls
    """
    
    # First try to extract from <tool_call> tags
    # Handle both actual newlines and escaped \n characters
    tool_call_pattern = r'<tool_call>[\s\\n]*(\{.*?\})[\s\\n]*</tool_call>'
    tool_calls = re.findall(tool_call_pattern, raw_prediction, re.DOTALL)
    
    # if debug and tool_calls:
    #     print(f"  Found {len(tool_calls)} tool_call(s)")
    #     for idx, tc in enumerate(tool_calls):
    #         print(f"    Tool call {idx + 1}: {tc[:100]}...")
    
    # If no <tool_call> tags found, try to find JSON in markdown code blocks
    if not tool_calls:
        # Look for ```json blocks or standalone JSON objects
        code_block_pattern = r'```(?:json)?[\s\\n]*(\{.*?\})[\s\\n]*```'
        code_blocks = re.findall(code_block_pattern, raw_prediction, re.DOTALL)
        
        if code_blocks:
            tool_calls = code_blocks
            if debug:
                print(f"  Found {len(code_blocks)} code block(s)")
        else:
            # Try to find standalone JSON objects (as a last resort)
            # Only if the text looks like it might contain a tool call
            if '"computer_use"' in raw_prediction and '"arguments"' in raw_prediction:
                json_pattern = r'\{[^{}]*"name"\s*:\s*"computer_use"[^{}]*"arguments"\s*:\s*\{[^}]*\}[^}]*\}'
                standalone_json = re.findall(json_pattern, raw_prediction, re.DOTALL)
                if standalone_json:
                    tool_calls = standalone_json
                    if debug:
                        print(f"  Found {len(standalone_json)} standalone JSON(s)")
    
    if not tool_calls:
        # No valid tool calls found - this is a refusal/explanation
        if debug:
            print("  No tool calls found - returning (None, None)")
        return None, None
    
    # Parse all actions
    actions = []
    for tool_call_str in tool_calls:
        try:
            # Clean up the JSON string - remove escaped newlines and extra whitespace
            tool_call_str = tool_call_str.strip()
            tool_call_str = tool_call_str.replace('\\n', '')
            tool_call_json = json.loads(tool_call_str)
            
            # Handle both direct arguments and nested structure
            if 'arguments' in tool_call_json:
                actions.append(tool_call_json['arguments'])
            elif 'action' in tool_call_json:
                # Direct action format (less common)
                actions.append(tool_call_json)
        except json.JSONDecodeError as e:
            if debug:
                print(f"  JSON decode error: {e}")
            # Skip malformed JSON
            continue
    
    if not actions:
        if debug:
            print("  No valid actions parsed - returning (None, None)")
        return None, None
    
    if debug and len(actions) > 1:
        print(f"  *** MULTIPLE ACTIONS DETECTED: {len(actions)} actions ***")
        for idx, action in enumerate(actions):
            print(f"    Action {idx + 1}: {action}")
    
    # Extract coordinates from first action that has them
    coordinates = None
    for action in actions:
        if 'coordinate' in action and action['coordinate']:
            coordinates = action['coordinate']
            # if debug:
            #     print(f"  Coordinates from action: {coordinates}")
            break
    
    # Determine action type: first non-click action, or first action if all are clicks
    click_actions = {'left_click', 'right_click', 'middle_click', 'double_click', 
                     'mouse_move', 'left_click_drag'}
    
    action_type = None
    non_click_action = None
    
    for action in actions:
        action_name = action.get('action', '')
        
        # Store first action as fallback
        if action_type is None:
            action_type = 'click'
        
        # Look for first non-click action
        if action_name not in click_actions:
            non_click_action = action_name
            if debug and len(actions) > 1:
                print(f"  Found non-click action: {non_click_action}")
            break
    
    # Use non-click action if found, otherwise use first action
    if non_click_action:
        action_type = non_click_action
    
    if debug and len(actions) > 1:
        print(f"-"*100+"\n")
        print(f"  Raw prediction: {raw_prediction}")
        print(f"-"*100+"\n")
        print(f"  Final action_type selected: {action_type}")
        print(f"  Final coordinates selected: {coordinates}\n")
        print(f"="*100+"\n")
    
    # Handle special cases like 'terminate' with status
    if action_type == 'terminate' and actions:
        for action in actions:
            if action.get('action') == 'terminate' and 'status' in action:
                status = action.get('status', '')
                if status:
                    action_type = f"terminate_{status}"
                break

    if action_type == 'browser_select_option':
        action_type = 'select'

    return action_type, coordinates


# Test with problematic examples
if __name__ == "__main__":
    examples = [
        # Example 1: Type action with escaped newlines (literal \n)
        r'Thought: The goal is to type \'nutritionist\' into the search box labeled \'What\'. The current input is \'nutritionist\', which matches the intended search term.\n<tool_call>\n{"name": "computer_use", "arguments": {"action": "type", "text": "nutritionist"}}\n</tool_call>',
        
        # Example 2: Scroll action with escaped newlines (literal \n)
        r'Thought: The goal is to click on the link above \'APPLY\'. Since there isn\'t a visible \'APPLY\' button in the current view, I\'ll need to scroll down to find it.\n<tool_call>\n{"name": "computer_use", "arguments": {"action": "scroll", "pixels": -400}}\n</tool_call>',
        
        # Example 3: JSON in code block
        '```json\n{"name": "computer_use", "arguments": {"action": "left_click", "coordinate": [510, 622]}}\n```',
        
        # Example 4: Simple tool call with real newlines
        '<tool_call>\n{"name": "computer_use", "arguments": {"action": "left_click", "coordinate": [966, 546]}}\n</tool_call>',
        
        # Example 5: Multiple actions with real newlines
        'Thought: I need to locate the textbox.\n<tool_call>\n{"name": "computer_use", "arguments": {"action": "left_click", "coordinate": [704, 468]}}\n</tool_call>\n<tool_call>\n{"name": "computer_use", "arguments": {"action": "type", "text": "SAN FRANCISCO"}}\n</tool_call>',
    ]
    
    for i, example in enumerate(examples, 1):
        print(f"\n{'='*60}")
        print(f"Example {i}:")
        print(f"  Raw (first 150 chars): {repr(example[:150])}")
        action_type, coords = parse_qwen_prediction(example, debug=True)
        print(f"\n  RESULT:")
        print(f"    Action type: {action_type}")
        print(f"    Coordinates: {coords}")
        print(f"    Valid action: {action_type is not None}")

In [ ]:
qwen_filter = df_all['model'] == 'qwen25vl'
qwen_df = df_all[qwen_filter].copy()
results = qwen_df['raw_prediction'].apply(parse_qwen_prediction)
qwen_df['action_type'] = results.apply(lambda x: x[0])
qwen_df['coordinates'] = results.apply(lambda x: x[1])
# qwen_df[['coordinates', 'action_type']].head()
print(qwen_df['coordinates'].isna().sum())
# qwen_df[qwen_df['coordinates'].isna()].head(20)
qwen_df.info()
qwen_df.groupby('action_type')[['model', 'use_reasoning', 'query_type', 'action_type']].value_counts()



## UI-TARS1.5

Parse for:
1. `Action: click(start_box='(681,385)')`
2. `Action: click(start_box='(1168,198]')`


In [ ]:
import ast

# UI-TARS1.5 action parsing functions
def parse_action(action_str):
    try:
        # 解析字符串为 AST 节点
        node = ast.parse(action_str, mode='eval')

        # 确保节点是一个表达式
        if not isinstance(node, ast.Expression):
            raise ValueError("Not an expression")

        # 获取表达式的主体
        call = node.body

        # 确保主体是一个函数调用
        if not isinstance(call, ast.Call):
            raise ValueError("Not a function call")

        # 获取函数名
        if isinstance(call.func, ast.Name):
            func_name = call.func.id
        elif isinstance(call.func, ast.Attribute):
            func_name = call.func.attr
        else:
            func_name = None

        # 获取关键字参数
        kwargs = {}
        for kw in call.keywords:
            key = kw.arg
            # 处理不同类型的值，这里假设都是常量
            if isinstance(kw.value, ast.Constant):
                value = kw.value.value
            elif isinstance(kw.value, ast.Str):  # 兼容旧版本 Python
                value = kw.value.s
            else:
                value = None
            kwargs[key] = value

        return {
            'function': func_name,
            'args': kwargs
        }

    except Exception as e:
        print(f"Failed to parse action '{action_str}': {e}")
        return None

def parse_action_to_structure_output(text, factor = IMAGE_FACTOR, origin_resized_height = 1080, origin_resized_width = 1920, model_type = "qwen25vl", max_pixels=16384*28*28, min_pixels=100*28*28):
    text = text.strip()
    if model_type == "qwen25vl":
        smart_resize_height, smart_resize_width = smart_resize(origin_resized_height, origin_resized_width, factor=IMAGE_FACTOR, min_pixels=min_pixels, max_pixels=max_pixels)

    # 正则表达式匹配 Action 字符串
    if text.startswith("Thought:"):
        thought_pattern = r"Thought: (.+?)(?=\s*Action:|$)"
        thought_hint = "Thought: "
    elif text.startswith("Reflection:"):
        thought_pattern = r"Reflection: (.+?)Action_Summary: (.+?)(?=\s*Action:|$)"
        thought_hint = "Reflection: "
    elif text.startswith("Action_Summary:"):
        thought_pattern = r"Action_Summary: (.+?)(?=\s*Action:|$)"
        thought_hint = "Action_Summary: "
    else:
        thought_pattern = r"Thought: (.+?)(?=\s*Action:|$)"
        thought_hint = "Thought: "
    reflection, thought = None, None
    thought_match = re.search(thought_pattern, text, re.DOTALL)
    if thought_match:
        if len(thought_match.groups()) == 1:
            thought = thought_match.group(1).strip()
        elif len(thought_match.groups()) == 2:
            thought = thought_match.group(2).strip()
            reflection = thought_match.group(1).strip()
    assert "Action:" in text
    action_str = text.split("Action:")[-1]

    tmp_all_action = action_str.split("\n\n")
    all_action = []
    for action_str in tmp_all_action:
        if "type(content" in action_str:
            # 正则表达式匹配 content 中的字符串并转义单引号
            def escape_quotes(match):
                content = match.group(1)  # 获取 content 的值
                return content

            # 使用正则表达式进行替换
            pattern = r"type\(content='(.*?)'\)"  # 匹配 type(content='...')
            content = re.sub(pattern, escape_quotes, action_str)

            # 处理字符串
            action_str = escape_single_quotes(content)
            action_str = "type(content='" + action_str + "')"
        all_action.append(action_str)

    parsed_actions = [parse_action(action.replace("\n","\\n").lstrip()) for action in all_action]
    actions = []
    for action_instance, raw_str in zip(parsed_actions, all_action):
        if action_instance == None:
            print(f"Action can't parse: {raw_str}")
            raise ValueError(f"Action can't parse: {raw_str}") 
        action_type = action_instance["function"]
        params = action_instance["args"]

        # import pdb; pdb.set_trace()
        action_inputs = {}
        for param_name, param in params.items():
            if param == "": continue
            param = param.lstrip()  # 去掉引号和多余的空格
            # 处理start_box或者end_box参数格式 '<bbox>x1 y1 x2 y2</bbox>'
            action_inputs[param_name.strip()] = param
            
            if "start_box" in param_name or "end_box" in param_name:
                ori_box = param
                # Remove parentheses and brackets, then split the string by commas
                numbers = ori_box.replace("(", "").replace(")", "").replace("[", "").replace("]", "").split(",")
                # Convert to float and scale by 1000
                # Qwen2.5vl output absolute coordinates, qwen2vl output relative coordinates
                if model_type == "qwen25vl":
                    float_numbers = []
                    for num_idx, num in enumerate(numbers):
                        num = float(num)
                        if (num_idx + 1) % 2 == 0:
                            float_numbers.append(round(float(num/smart_resize_height * IMAGE_HEIGHT)))
                        else:
                            float_numbers.append(round(float(num/smart_resize_width * IMAGE_WIDTH)))
                else:
                    raise ValueError(f"Unknown model type: {model_type}")

                # if len(float_numbers) == 2:
                #     float_numbers = [float_numbers[0], float_numbers[1], float_numbers[0], float_numbers[1]]
                action_inputs[param_name.strip()] = str(float_numbers)

        # import pdb; pdb.set_trace()
        actions.append({
            "reflection": reflection,
            "thought": thought,
            "action_type": action_type,
            "action_inputs": action_inputs,
            "text": text
        })
    return actions




In [ ]:
uitars15_filter = df_all['model'] == 'uitars15'
uitars15_df = df_all[uitars15_filter].copy()

uitars15_df['structured_actions'] = uitars15_df['raw_prediction'].apply(parse_action_to_structure_output)  # use qwen25vl for uitars15
pd.set_option('display.max_columns', None)

# def has_more_than_1_action(row):
#     return True if len(row['structured_actions']) > 1 else False

# print("rows with more than 1 action:")
# display(len(uitars15_df[uitars15_df.apply(has_more_than_1_action, axis=1)]))


# def has_non_click_action(row):
#     return True if any(action['action_type'] != 'click' for action in row['structured_actions']) else False

# display(uitars15_df[uitars15_df.apply(has_non_click_action, axis=1)][['raw_prediction', 'structured_actions']].head())

def get_action_type_and_coordinates_from_structured_actions_for_uitars15(row):
    structured_actions = row['structured_actions']
    action_type = structured_actions[0]['action_type']
    if action_type == 'click':
        coordinates = structured_actions[0]['action_inputs']['start_box']
        coordinates = ast.literal_eval(coordinates)
        return action_type, coordinates

    else:
        # from pprint import pprint
        # pprint(structured_actions)
        coordinates = None
    return action_type, coordinates

uitars15_df['action_type'], uitars15_df['coordinates'] = zip(*uitars15_df.apply(get_action_type_and_coordinates_from_structured_actions_for_uitars15, axis=1))
uitars15_df.groupby('action_type')[['model', 'use_reasoning', 'query_type', 'action_type']].value_counts()
uitars15_df[uitars15_df['action_type'] == 'click'].head()

## GTA1

Parse for:
1. `Action: click(start_box='(648,409)')`
2. 
    ```
    Thought: I noticed a search box at the top of the page, which is perfect for looking up pork. I need to click on it first to make it active so that I can enter the keyword.
    Action: (583,287)
    ```
3. `(1444,42)`
4. `-203,240`
5. 
    ```
    Thought: I noticed the link for Audiology, which is located just below the Allergies & Immunology link. It's positioned in the left column, and I need to click on it to access the relevant page.
    Action: (389,546)
    assistant
    Thought: I'm currently looking at the list of medical specialties, and I've found the link for Anesthesiology. It's located in the left column, just below the Allergy & Immunology link. I need to click on it to learn more about this field.
    Action: (408,503)
    assistant
    Thought: I'm currently browsing through the list of medical specialties and I've come across the link for Audiology. It's positioned in the left column, just below the Allergy & Immunology link. I need to click on it to get more information about this field.
    Action: (389,546)
    assistant
    Thought: I just clicked on the link for Audiology, and now I need to move on to the next link. I see that the link for Anesthesiology is right below it, so I'll click on that one next. This way, I can continue exploring the various medical specialties.
    Action: (408,503)
    assistant
    Thought: I just clicked on the link for Audiology, and now I need to move on to the next link. I see that the link for Anesthesiology is right below it, so I'll click on that one next. This way, I can continue exploring the various medical specialties.
    Action: (408,503)
    assistant
    Thought: I just clicked on the link for Audiology, and now I need to move on to the next link. I see that the link for Anesthesiology is right below it, so I'll click on that one next. This way, I can continue exploring the various medical specialties.
    Action: (408,503)
    assistant
    ...
    ```

In [ ]:
def extract_coordinates(raw_string):
    try:
        # First try to match coordinates with parentheses: (x,y) or (x, y)
        matches = re.findall(r"\((-?\d*\.?\d+),\s*(-?\d*\.?\d+)\)", raw_string)
        if matches:
            return [int(matches[0][0]), int(matches[0][1])]
        
        # If no parentheses match, try without parentheses: x,y or -x,y
        matches = re.findall(r"(-?\d+\.?\d*),\s*(-?\d+\.?\d*)", raw_string)
        if matches:
            return tuple(map(int, matches[0]))
        
        return [0, 0]
    except:
        return [0, 0]


In [ ]:
gta1_filter = df_all['model'] == 'gta1'
gta1_df = df_all[gta1_filter].copy()
gta1_df['coordinates'] = gta1_df['raw_prediction'].apply(extract_coordinates)
smart_resize_height, smart_resize_width = smart_resize(IMAGE_HEIGHT, IMAGE_WIDTH, factor=IMAGE_FACTOR, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
print(smart_resize_height, smart_resize_width)
gta1_df['coordinates'] = gta1_df['coordinates'].apply(lambda x: [x[0] / smart_resize_height * IMAGE_HEIGHT, x[1] / smart_resize_width * IMAGE_WIDTH])

# Check for anomalies (na, malformed, etc)

In [ ]:
uitars15_df.isna().sum()

# Denormalize all coordinates

UITARS and Qwen should have denormalization logic in their parsing logic, but double check. GTA1 would need denormalization

# Calculate action str exact match

Since GTA1 was trained to output coordinates only, it's not very meaningful to compare action str exact match.

In [ ]:
def is_action_type_correct(row):
    gt_action_type = row['instruction'].strip().split(' ')[0].lower()
    if gt_action_type not in ['click', 'type', 'select', 'scroll']:
        print(f"Action type not in ['click', 'type', 'select', 'scroll']: {gt_action_type}")
        return 0
    if row['action_type'] == gt_action_type:
        return 1
    elif row['action_type'] in ['type', 'click'] and gt_action_type == 'select':
        return 1
    else:
        # print(f"Action type mismatch: {action_type} != {gt_action_type}")
        return 0

# qwen_df[(qwen_df['action_exact_match'] == 0)].head()
display(qwen_df.apply(is_action_type_correct, axis=1).sum() / qwen_df.shape[0])

# uitars15_df[(uitars15_df['action_exact_match'] == 0)].head()
display(uitars15_df.apply(is_action_type_correct, axis=1).sum() / uitars15_df.shape[0])



# Calculate hit box accuracy

In [ ]:
import ast

def is_coords_in_bbox(coords, bbox):
    if coords is None:
        return 0

    tolerance = 20
    x1, y1, w, h = ast.literal_eval(bbox)
    x, y = coords
    return x1 - tolerance / 2 <= x <= x1 + w + tolerance / 2 and y1 - tolerance / 2 <= y <= y1 + h + tolerance / 2

# 1. 2d -> is_coords_in_bbox(coords_2d, gt_bbox)
# 2. 4d -> get center from the 4d coordinates
# 3. extra coordinates, just skip and use coords_4d -> step 2
# 4. non-coordinate actions when it's 'type', or malformed. How do we handle this?  

# 1. penalize malformed & actions requiring coordinates & wrong actions (finished, wait, scroll)
# 2. consider type and select without coordinates as 0 (instruction doesn't explicitly say predict coordinates)

# 

def is_bbox_hit(row):
    action_type = row.get('action_type', None)
    pred = row.get('coordinates')
    gt_bbox = row.get('ground_truth_bbox')

    if action_type == 'click':
        return is_coords_in_bbox(pred, gt_bbox)
    elif action_type == 'scroll':
        return is_coords_in_bbox(pred, gt_bbox)
    elif action_type == 'type':
        return is_coords_in_bbox(pred, gt_bbox)
    elif action_type == 'select':
        return is_coords_in_bbox(pred, gt_bbox)
    elif action_type == 'wait':
        return 0
    elif action_type == 'finished':
        return 0
    elif action_type == 'call_user':
        return 0
    elif action_type == None:
        try:
            return is_coords_in_bbox(pred, gt_bbox)
        except:
            return 0
    else:
        raise ValueError(f"Unknown action type: {action_type}")


qwen_df['hit_box_accuracy'] = qwen_df.apply(is_bbox_hit, axis=1)
print(qwen_df[~qwen_df['action_type'].isin(['click', 'select', 'scroll', 'type'])].count())

uitars15_df['hit_box_accuracy'] = uitars15_df.apply(is_bbox_hit, axis=1)
print(uitars15_df[~uitars15_df['action_type'].isin(['click', 'select', 'scroll', 'type'])].count())

gta1_df['hit_box_accuracy'] = gta1_df.apply(is_bbox_hit, axis=1)



In [ ]:
display(qwen_df[qwen_df['query_type'] == 'direct_query']['hit_box_accuracy'].sum() / qwen_df[qwen_df['query_type'] == 'direct_query'].shape[0])
display(uitars15_df[uitars15_df['query_type'] == 'direct_query']['hit_box_accuracy'].sum() / uitars15_df[uitars15_df['query_type'] == 'direct_query'].shape[0])
display(gta1_df[gta1_df['query_type'] == 'direct_query']['hit_box_accuracy'].sum() / gta1_df[gta1_df['query_type'] == 'direct_query'].shape[0])

display(qwen_df[qwen_df['query_type'] == 'relational_query']['hit_box_accuracy'].sum() / qwen_df[qwen_df['query_type'] == 'relational_query'].shape[0])
display(uitars15_df[uitars15_df['query_type'] == 'relational_query']['hit_box_accuracy'].sum() / uitars15_df[uitars15_df['query_type'] == 'relational_query'].shape[0])
display(gta1_df[gta1_df['query_type'] == 'relational_query']['hit_box_accuracy'].sum() / gta1_df[gta1_df['query_type'] == 'relational_query'].shape[0])

# Combine dfs

In [ ]:
df = pd.concat([gta1_df, qwen_df, uitars15_df])

In [ ]:
# Only UI-TARS15 has structured_actions from our parsing logic
# Only UI-TARS15 and Qwen2.5 have action_type

df.info()


# Calculate bbox center MSE & NSME

In [ ]:
"""
1. get nmse for normal coordinates

For None coords_2d
2. calculate max mse & nmse from the rows with normal coordinates
3. use 95% quantile of max mse and max nmse, as the threshold for NMSE_MALFORM_PENALTY
4. Add min-max normalization for MSE using dataset statistics
5. Add z-score normalization option for MSE using mean and std
"""

# Choose normalization method: 'minmax' or 'zscore'
NORMALIZATION_METHOD = 'minmax'  # Change to 'zscore' to use z-score normalization

def get_bbox_mse_and_normalized_mse(pred, gt_bbox):
    """
    Calculate the normalized mean squared error of the bbox center

    Args:
        pred: (x, y)
        gt_bbox: (x1, y1, w, h)
    Returns:
        (mse, normalized_mse)
    """
    x1, y1, w, h = ast.literal_eval(gt_bbox)
    diagonal_squared = w ** 2 + h ** 2

    if pred is None:
        return None, None  # calculate later

    x, y = pred
    
    # Validate coordinates are within image bounds
    # For 1920x1080 image, coordinates should be in [0, 1920] x [0, 1080]
    MAX_VALID_MSE = IMAGE_WIDTH ** 2 + IMAGE_HEIGHT ** 2  # ~4.85M for 1920x1080
    
    # Check if coordinates are out of bounds (indicates parsing/scaling error)
    if x < 0 or x > IMAGE_WIDTH or y < 0 or y > IMAGE_HEIGHT:
        # Coordinates are invalid - return None to be handled as malformed
        return None, None
    
    gt_center = (x1 + w / 2, y1 + h / 2) # gt_bbox follows the format of (x1, y1, w, h)
    mse = ((x - gt_center[0]) ** 2 + (y - gt_center[1]) ** 2)
    
    # Additional validation: if MSE is impossibly large, treat as malformed
    if mse > MAX_VALID_MSE * 2:  # Allow some margin (2x) for edge cases
        return None, None
    
    return mse, mse # mse / diagonal_squared

# Apply function and expand results into DataFrame columns
mse_results = df.apply(lambda x: get_bbox_mse_and_normalized_mse(x['coordinates'], x['ground_truth_bbox']), axis=1, result_type='expand')
df['bbox_center_mse'] = mse_results[0]
df['normalized_mse'] = mse_results[1]

# Validate MSE values - maximum possible MSE for 1920x1080 image is ~4.85M
MAX_VALID_MSE = IMAGE_WIDTH ** 2 + IMAGE_HEIGHT ** 2  # 1920^2 + 1080^2 = 4,852,800
invalid_mse_mask = (df['bbox_center_mse'] > MAX_VALID_MSE * 2) & df['bbox_center_mse'].notna()

if invalid_mse_mask.sum() > 0:
    print(f"\nWARNING: Found {invalid_mse_mask.sum()} rows with impossibly large MSE values (> {MAX_VALID_MSE * 2:.0f})")
    print("These likely indicate coordinate parsing/scaling errors. Sample invalid rows:")
    invalid_rows = df[invalid_mse_mask][['model', 'coordinates', 'bbox_center_mse']].head(10)
    print(invalid_rows)
    print("\nTreating these as malformed (setting to None)...")
    # Set invalid MSE values to None (will be handled as malformed)
    df.loc[invalid_mse_mask, 'bbox_center_mse'] = None
    df.loc[invalid_mse_mask, 'normalized_mse'] = None

# Get rows with normal coordinates (where we have valid mse values)
normal_coords_mask = df['bbox_center_mse'].notna() & df['normalized_mse'].notna()
normal_coords_df = df[normal_coords_mask]

print(f"\nRows with normal coordinates: {normal_coords_mask.sum()}")
print(f"Rows with None coordinates: {(~normal_coords_mask).sum()}")

# Print rows with invalid/None coordinates for inspection
invalid_coords_rows = df[~normal_coords_mask]
if len(invalid_coords_rows) > 0:
    print(f"\nRows with invalid/None coordinates (showing first 50):")
    # Show key columns including instruction and raw_prediction
    cols_to_show = ['model', 'variant', 'use_reasoning', 'query_type', 'instruction', 'raw_prediction', 'coordinates', 'bbox_center_mse', 'normalized_mse']
    print(invalid_coords_rows[cols_to_show].head(50).to_string())
    if len(invalid_coords_rows) > 50:
        print(f"\n... and {len(invalid_coords_rows) - 50} more rows with invalid coordinates")
else:
    print("\nNo rows with invalid coordinates found.")

# Calculate 95% quantile of mse and normalized_mse from rows with normal coordinates
MSE_MALFORM_PENALTY = normal_coords_df['bbox_center_mse'].quantile(0.95)
NMSE_MALFORM_PENALTY = normal_coords_df['normalized_mse'].quantile(0.95)

print(f"\n95% quantile MSE: {MSE_MALFORM_PENALTY:.4f}")
print(f"95% quantile NMSE: {NMSE_MALFORM_PENALTY:.4f}")

# Calculate min-max normalization parameters for MSE
# Use 0 as min (perfect match) and 95th percentile as max (consistent with malformed penalty)
# This ensures most values are in [0, 1] range and outliers don't dominate
MSE_MIN = 0.0  # Perfect prediction
MSE_MAX = MSE_MALFORM_PENALTY  # Use 95th percentile as max for min-max normalization

# Also calculate min-max for normalized_mse (diagonal-normalized)
NMSE_MIN = 0.0  # Perfect prediction
NMSE_MAX = NMSE_MALFORM_PENALTY  # Use 95th percentile as max

print(f"\nMin-Max Normalization Parameters:")
print(f"  MSE: min={MSE_MIN:.4f}, max={MSE_MAX:.4f}")
print(f"  NMSE: min={NMSE_MIN:.4f}, max={NMSE_MAX:.4f}")

# Calculate z-score normalization parameters (mean and std) for MSE
MSE_MEAN = normal_coords_df['bbox_center_mse'].mean()
MSE_STD = normal_coords_df['bbox_center_mse'].std()
NMSE_MEAN = normal_coords_df['normalized_mse'].mean()
NMSE_STD = normal_coords_df['normalized_mse'].std()

print(f"\nZ-Score Normalization Parameters:")
print(f"  MSE: mean={MSE_MEAN:.4f}, std={MSE_STD:.4f}")
print(f"  NMSE: mean={NMSE_MEAN:.4f}, std={NMSE_STD:.4f}")

# Apply min-max normalization to MSE values
def apply_minmax_normalization_mse(mse_val, min_val, max_val):
    """Apply min-max normalization: (value - min) / (max - min)"""
    if mse_val is None:
        return None
    if max_val == min_val:
        return 0.0  # All values are the same
    # Clip values to [min, max] range before normalization
    clipped_val = max(min_val, min(mse_val, max_val))
    return (clipped_val - min_val) / (max_val - min_val)

# Apply z-score normalization to MSE values
def apply_zscore_normalization_mse(mse_val, mean_val, std_val):
    """Apply z-score normalization: (value - mean) / std"""
    if mse_val is None:
        return None
    if std_val == 0:
        return 0.0  # All values are the same
    return (mse_val - mean_val) / std_val

# Apply the chosen normalization method
print(f"\nUsing normalization method: {NORMALIZATION_METHOD}")
if NORMALIZATION_METHOD == 'zscore':
    df['normalized_mse'] = df['normalized_mse'].apply(
        lambda x: apply_zscore_normalization_mse(x, NMSE_MEAN, NMSE_STD)
    )
else:  # default to minmax
    df['normalized_mse'] = df['normalized_mse'].apply(
        lambda x: apply_minmax_normalization_mse(x, NMSE_MIN, NMSE_MAX)
    )

# Calculate total and average mse & nmse and use MSE_MALFORM_PENALTY and NMSE_MALFORM_PENALTY to fill in the None values
total_mse = normal_coords_df['bbox_center_mse'].sum() + (~normal_coords_mask).sum() * MSE_MALFORM_PENALTY
total_nmse = normal_coords_df['normalized_mse'].sum() + (~normal_coords_mask).sum() * NMSE_MALFORM_PENALTY
avg_mse = total_mse / len(df)
avg_nmse = total_nmse / len(df)

print(f"\nTotal MSE: {total_mse:.4f}")
print(f"Total NMSE: {total_nmse:.4f}")
print(f"Average MSE: {avg_mse:.4f}")
print(f"Average NMSE: {avg_nmse:.4f}")


### Per model

In [ ]:
df.info()

In [ ]:
# Display mean and sum at each groupby level
groupby_cols = ['model', 'use_reasoning', 'query_type', 'variant']

print("="*80)
print("MEAN of normalized_mse:")
print("="*80)
display(df.groupby(groupby_cols)['normalized_mse'].mean())

print("\n" + "="*80)
print("SUM at each groupby level:")
print("="*80)

# Sum at each level
for level in range(len(groupby_cols)):
    level_cols = groupby_cols[:level+1]
    print(f"\nSum by {level_cols}:")
    display(df.groupby(level_cols)['normalized_mse'].sum())
    
print("\n" + "="*80)
print("Grand Total:")
print("="*80)
print(f"Total sum of normalized_mse: {df['normalized_mse'].sum():.4f}")


# Calculate GIoU & Normalized GIoU with adaptive proxy bbox for predicted 2D coordinates

In [ ]:
def get_bbox_area(bbox):
    x1, y1, w, h = ast.literal_eval(bbox)
    return w * h

print(df['ground_truth_bbox'].apply(get_bbox_area).median())
print(df['ground_truth_bbox'].apply(get_bbox_area).describe())



In [ ]:
import math

reference_size = math.sqrt(df['ground_truth_bbox'].apply(get_bbox_area).median())
reference_size

In [ ]:
import numpy as np

# Convert 2D point prediction to 4D bbox prediction using ground truth size
def giou_point_to_bbox(row, alpha=0.5, min_proxy=16.0, max_proxy=128.0):
    # unpack ground truth
    x_gt, y_gt, w_gt, h_gt = ast.literal_eval(row["ground_truth_bbox"])
    
    # handle coordinates - can be None, list, tuple, or string
    coords = row["coordinates"]
    if coords is None:
        return np.nan, np.nan
    
    # if it's a string, parse it
    if isinstance(coords, str):
        try:
            coords = ast.literal_eval(coords)
        except:
            return np.nan, np.nan
    
    # ensure coordinates can be unpacked (must be iterable with 2 elements)
    try:
        px, py = coords
    except (TypeError, ValueError):
        return np.nan, np.nan
    
    # Validate coordinates are within image bounds (0 to IMAGE_WIDTH for x, 0 to IMAGE_HEIGHT for y)
    if px < 0 or px > IMAGE_WIDTH or py < 0 or py > IMAGE_HEIGHT:
        # Coordinates are outside valid image bounds - return NaN
        return np.nan, np.nan

    # ground truth box: (x1, y1, x2, y2)
    gt = np.array([x_gt, y_gt, x_gt + w_gt, y_gt + h_gt])

    # Use ground truth bounding box size directly (convert 2D prediction to 4D)
    pw = w_gt
    ph = h_gt

    # predicted box centered at (px, py) with ground truth size
    pb = np.array([
        px - pw / 2,
        py - ph / 2,
        px + pw / 2,
        py + ph / 2
    ])

    # intersection
    ix1, iy1 = np.maximum(gt[:2], pb[:2])
    ix2, iy2 = np.minimum(gt[2:], pb[2:])
    iw = np.maximum(ix2 - ix1, 0.0)
    ih = np.maximum(iy2 - iy1, 0.0)
    inter = iw * ih

    # areas
    area_gt = w_gt * h_gt
    area_pb = pw * ph
    union = area_gt + area_pb - inter

    iou = inter / union if union > 0 else 0.0

    # smallest enclosing box
    cx1, cy1 = np.minimum(gt[:2], pb[:2])
    cx2, cy2 = np.maximum(gt[2:], pb[2:])
    area_c = (cx2 - cx1) * (cy2 - cy1)

    giou = iou - (area_c - union) / area_c
    ngiou = (giou + 1.0) / 2.0

    return giou, ngiou


In [ ]:
df['giou'], df['ngiou'] = zip(*df.apply(giou_point_to_bbox, axis=1))


In [ ]:
display(df.groupby(['model', 'use_reasoning', 'query_type', 'variant'])['giou'].mean().sort_values(ascending=True))
display(df.groupby(['model', 'use_reasoning', 'query_type', 'variant'])['ngiou'].mean().sort_values(ascending=True))


# Save df all

In [ ]:
# pd.concat([gta1_df, qwen_df, uitars15_df]).to_csv('baseline_results_combined.csv', index=False)
df.to_csv('baseline_results_full_new.csv', index=False)


# Load cleaned df

In [ ]:
# Use original CSV for metrics (cleaned CSV has corrupted hit_box_accuracy in some rows)
df = pd.read_csv('/Users/lockewang/FIG/WebDomainRandomizer/data/baseline_results_full_new_cleaned.csv')
clean = pd.read_csv('baseline_results_full_new_cleaned.csv', usecols=['interesting_cases'])
assert len(df) == len(clean), "Original and cleaned CSV row counts must match"
df['interesting_cases'] = clean['interesting_cases'].values
display(df['interesting_cases'].value_counts())
df = df[df['interesting_cases'] != 'Invalid'].copy()

In [ ]:
# --- DEBUG: run after the load cell to check filtered df ---
# If hit_box_accuracy is all 0/False here, the issue is load/filter or CSV row order.
print("DEBUG load/filter (df = filtered dataframe in memory):")
print("  df.shape:", df.shape)
print("  df['hit_box_accuracy'] dtype:", df['hit_box_accuracy'].dtype)
vc = df['hit_box_accuracy'].value_counts(dropna=False)
print("  value_counts (raw):", vc.head(10).to_dict())
print("  count True/1:", ((df['hit_box_accuracy'] == True) | (df['hit_box_accuracy'] == 1)).sum())
print("  count False/0:", ((df['hit_box_accuracy'] == False) | (df['hit_box_accuracy'] == 0)).sum())
print("--- end DEBUG ---")

# Plotting trends

In [ ]:
df['interesting_cases'].value_counts()

In [ ]:
df.info()

In [ ]:
# ============================================================================
# DATA PREPARATION FOR VISUALIZATIONS
# ============================================================================

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set style: whitegrid, axes and text black
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10
plt.rcParams['text.color'] = 'black'
plt.rcParams['axes.labelcolor'] = 'black'
plt.rcParams['axes.edgecolor'] = 'black'
plt.rcParams['xtick.color'] = 'black'
plt.rcParams['ytick.color'] = 'black'
plt.rcParams['legend.edgecolor'] = 'black'
plt.rcParams['axes.facecolor'] = '#f2f2f2'
plt.rcParams['figure.facecolor'] = '#f2f2f2'

# Create level column (L1 = direct_query, L2 = relational_query)
df['level'] = df['query_type'].map({
    'direct_query': 'L1',
    'relational_query': 'L2'
})

# Ensure hit_box_accuracy is numeric (CSV has "TRUE"/"FALSE" strings; to_numeric would coerce them to NaN)
if df['hit_box_accuracy'].dtype == object:
    df['hit_box_accuracy'] = df['hit_box_accuracy'].replace(
        {'TRUE': 1, 'True': 1, True: 1, 'FALSE': 0, 'False': 0, False: 0}
    )
df['hit_box_accuracy'] = pd.to_numeric(df['hit_box_accuracy'], errors='coerce')

# Single summary dataframe: group by query_type, then add level for delta/gaps
summary_df = df.groupby(['model', 'variant', 'use_reasoning', 'query_type']).agg({
    'hit_box_accuracy': 'mean',
    'bbox_center_mse': 'mean',
    'normalized_mse': 'mean',
    'giou': 'mean',
    'ngiou': 'mean'
}).reset_index()
summary_df['level'] = summary_df['query_type'].map({
    'direct_query': 'L1',
    'relational_query': 'L2'
})
summary_df['query_label'] = summary_df['query_type'].map({
    'direct_query': 'Direct',
    'relational_query': 'Relational'
})
summary_df['reasoning_label'] = summary_df['use_reasoning'].map({
    True: 'Reasoning',
    False: 'No Reasoning'
})
summary_df['model_label'] = summary_df['model'].str.upper()

# Metric labels for all visualizations
metric_labels = {
    'hit_box_accuracy': 'Target Hit Rate',
    'bbox_center_mse': 'Bbox Center MSE',
    'normalized_mse': 'Normalized MSE (NMSE)',
    'giou': 'GIoU',
    'ngiou': 'NGIoU'
}

# Shared names and orders for all plots
metrics = ['hit_box_accuracy', 'bbox_center_mse', 'normalized_mse', 'giou', 'ngiou']
models = sorted(df['model'].unique())
model_order = models
variants = ['original', 'precision', 'style', 'text_shrink']
# Dataset variant labels: same samples, different perturbation for evaluation
variant_labels = {
    'original': 'Unperturbed',
    'precision': 'Precision perturbation',
    'style': 'Style perturbation',
    'text_shrink': 'Text shrink perturbation'
}
# Configuration labels: query type + reasoning (no abbreviations)
config_order = [
    ('direct_query', True, 'Direct query\nWith reasoning'),
    ('direct_query', False, 'Direct query\nNo reasoning'),
    ('relational_query', True, 'Relational query\nWith reasoning'),
    ('relational_query', False, 'Relational query\nNo reasoning')
]

# Bar colors: dreamy + fig.inc-style — seaborn "pastel" (soft, clean, minimal-tech feel)
model_color_dict = {m: sns.color_palette("pastel", n_colors=len(models))[i] for i, m in enumerate(models)}
variant_color_dict = dict(zip(variants, sns.color_palette("pastel", n_colors=4)))

print("Data prepared for visualizations!")
print(f"Summary shape: {summary_df.shape}")
print(f"Models: {list(models)}")
print(f"Variants: {variants}")
print(f"Levels: L1, L2")


In [ ]:
summary_df.head()

# Primary Visualizations



## Visualization Requirements

- **Maximum subplots**: 2×2 subplots per figure
- **Annotations**: Numbers on bars/points (2-3 decimal places for readability)
- **Legends**: Outside plot area (right side), no overlap with content
- **Color schemes**: Consistent across all plots
  - Models: Fixed 3-color palette (Set2)
  - Variants: Fixed 4-color palette (Set3)
  - Configurations: Fixed 4-color palette (Set1)
- **Y-axis ranges**: 
  - GIoU: -1 to +1 with reference line at 0
  - Other metrics: Start at 0, auto-scale or log scale if needed for close values
- **Figure size**: Optimized for 2-column LaTeX papers (14-18" width, 8-12" height)
- **Layout spacing**: `tight_layout` with right margin for legends (rect=[0, 0, 0.92, 1])


In [ ]:
# ============================================================================
# HELPER FUNCTIONS FOR VISUALIZATIONS
# ============================================================================

from scipy import stats
import matplotlib.patches as mpatches

def calculate_ci(data, confidence=0.95, metric_name=None):
    """
    Calculate 95% confidence interval using t-distribution
    
    Args:
        data: Array-like data
        confidence: Confidence level (default 0.95)
        metric_name: Name of metric for bounds checking ('giou', 'ngiou', 'mse', 'nmse', etc.)
    
    Returns:
        (ci_lower, ci_upper) tuple
    """
    if len(data) < 2:
        return np.nan, np.nan
    data_clean = data.dropna() if hasattr(data, 'dropna') else np.array(data)[~np.isnan(data)]
    if len(data_clean) < 2:
        return np.nan, np.nan
    
    # Define metric bounds
    metric_bounds = {
        'giou': (-1.0, 1.0),      # GIoU ranges from -1 to 1
        'ngiou': (0.0, 1.0),      # NGIoU ranges from 0 to 1
        'hit_box_accuracy': (0.0, 1.0),  # Accuracy ranges from 0 to 1
        'mse': (0.0, np.inf),     # MSE is unbounded above, but >= 0
        'nmse': (0.0, np.inf),    # NMSE is unbounded above, but >= 0
        'bbox_center_mse': (0.0, np.inf),
        'normalized_mse': (0.0, np.inf)
    }
    
    mean = np.mean(data_clean)
    sem = stats.sem(data_clean, nan_policy='omit')
    h = sem * stats.t.ppf((1 + confidence) / 2., len(data_clean) - 1)
    
    ci_lower = mean - h
    ci_upper = mean + h
    
    # Clip confidence intervals to valid bounds for bounded metrics
    if metric_name and metric_name.lower() in metric_bounds:
        min_bound, max_bound = metric_bounds[metric_name.lower()]
        ci_lower = max(ci_lower, min_bound)
        ci_upper = min(ci_upper, max_bound)
    
    return ci_lower, ci_upper

def calculate_mean_and_ci(df, metric, groupby_cols):
    """Calculate mean and confidence intervals for a metric grouped by columns"""
    results = []
    for name, group in df.groupby(groupby_cols):
        if isinstance(name, tuple):
            group_dict = dict(zip(groupby_cols, name))
        else:
            group_dict = {groupby_cols[0]: name}
        
        values = group[metric].dropna()
        if len(values) > 0:
            # Pass metric name for bounds checking
            ci_lower, ci_upper = calculate_ci(values, metric_name=metric)
            group_dict.update({
                'mean': np.nanmean(values),
                'ci_lower': ci_lower,
                'ci_upper': ci_upper,
                'std': np.nanstd(values),
                'count': len(values)
            })
            results.append(group_dict)
    
    return pd.DataFrame(results)

print("Helper functions loaded!")



# Overall Grouped Bar Plot

In [ ]:
# ============================================================================
# PLOT 1.1: Model Performance by Variant (augmentation comparison)
# One figure: 4 rows × 1 column; Direct+no reasoning first (reference); shared legend & axis labels
# ============================================================================

metric = 'hit_box_accuracy'
use_pct = (metric == 'hit_box_accuracy')

# Config order: Direct + no reasoning first as reference, then others
config_order_plot = [
    ('direct_query', False, 'Direct query\nNo reasoning'),
    ('direct_query', True, 'Direct query\nWith reasoning'),
    ('relational_query', True, 'Relational query\nWith reasoning'),
    ('relational_query', False, 'Relational query\nNo reasoning')
]

fig, axes = plt.subplots(4, 1, figsize=(10, 16))
if not isinstance(axes, np.ndarray):
    axes = np.array([axes])
axes = axes.flatten()

# Y-axis: start well above 0 so differences are visually clear (e.g. 28–94% range uses most of vertical space)
all_pct_vals = []
for variant in variants:
    vdata = summary_df[summary_df['variant'] == variant]
    for _, row in vdata.iterrows():
        val = row[metric]
        if not pd.isna(val):
            all_pct_vals.append(val * 100 if use_pct else val)
y_min_global = 0.0
if use_pct and all_pct_vals:
    data_min = min(all_pct_vals)
    # Floor at 20% so axis never starts at 0; small padding below min so bars aren't cut
    y_min_global = max(20, data_min - 5)
    y_min_global = (y_min_global // 5) * 5  # round down to nearest 5 for clean ticks

fig.suptitle(f'Cross-Model Target Element Hit Accuracy', fontsize=18, fontweight='bold', y=0.995)
fig.supxlabel('Configuration', fontsize=18, fontweight='bold')
fig.supylabel(f'Accuracy (%)' if use_pct else metric_labels[metric], fontsize=18, fontweight='bold')

legend_handles, legend_labels = None, None

for variant_idx, variant in enumerate(variants):
    ax = axes[variant_idx]
    variant_data = summary_df[summary_df['variant'] == variant].copy()
    x_pos = np.arange(len(config_order_plot))
    width = 0.25
    bar_positions = [x_pos - width, x_pos, x_pos + width]

    all_values = []
    for model in models:
        values = []
        for query_type, reasoning, _ in config_order_plot:
            model_data = variant_data[
                (variant_data['model'] == model) &
                (variant_data['query_type'] == query_type) &
                (variant_data['use_reasoning'] == reasoning)
            ]
            if len(model_data) > 0:
                val = model_data[metric].iloc[0]
                values.append(val if not pd.isna(val) else 0)
            else:
                values.append(0)
        all_values.append(values)

    if use_pct:
        plot_vals = [[round(v * 100, 1) for v in row] for row in all_values]
    else:
        plot_vals = all_values

    # Horizontal dashed lines at direct+no reasoning (reference) for each model
    for model_idx, model in enumerate(models):
        ref_val = plot_vals[model_idx][0]
        ax.axhline(y=ref_val, color=model_color_dict[model], linestyle='--', alpha=0.6, linewidth=1.5, zorder=0)

    bars = []
    for model_idx, model in enumerate(models):
        b = ax.bar(bar_positions[model_idx], plot_vals[model_idx], width,
                   label=model.upper(), color=model_color_dict[model], alpha=0.85, edgecolor='none', zorder=1)
        bars.append(b)
        for bar, val in zip(bars[model_idx], plot_vals[model_idx]):
            if val > 0 or (metric == 'giou' and val < 0):
                height = bar.get_height()
                lbl = f'{val:.1f}%' if use_pct else f'{val:.3f}'
                ax.text(bar.get_x() + bar.get_width()/2., height, lbl, ha='center',
                        va='bottom' if height >= 0 else 'top', fontsize=12, fontweight='bold', zorder=2)

    ax.set_title('')
    ax.set_xticks(x_pos)
    ax.set_xticklabels([label for _, _, label in config_order_plot], fontsize=12)
    ax.grid(True, alpha=0.3, linestyle='--', axis='y', zorder=0)
    if use_pct:
        ax.set_ylim(bottom=y_min_global, top=100)
    else:
        ax.set_ylim(bottom=y_min_global if all_pct_vals else 0)
    if variant_idx == 0:
        legend_handles, legend_labels = ax.get_legend_handles_labels()

fig.legend(legend_handles, legend_labels, ncol=3, fontsize=12, loc='lower center',
           bbox_to_anchor=(0.5, 0.94), framealpha=0.95)
plt.tight_layout(rect=[0.03, 0.02, 0.90, 0.94])
plt.subplots_adjust(hspace=0.4)
# Place subplot titles in figure coordinates so they align with main title and legend (all centered at 0.5)
for variant_idx, ax in enumerate(axes):
    pos = ax.get_position()
    fig.text(0.5, pos.y1 + 0.008, variant_labels[variants[variant_idx]], fontsize=14, fontweight='bold',
             ha='center', va='bottom')
plt.show()
